In [1]:
#BLOCK 0 — Imports & Check

In [10]:
import re, sys, site
from itertools import islice
import transformers, datasets, huggingface_hub
from haystack.document_stores import FAISSDocumentStore
import faiss
import sentence_transformers
import torch
from datasets import load_dataset
from transformers.modeling_utils import SequenceSummary
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from haystack.document_stores import FAISSDocumentStore
from haystack.nodes import DensePassageRetriever
from haystack import Document


print("CUDA:", torch.cuda.is_available())
print("Torch:", torch.__version__)


CUDA: False
Torch: 2.9.1+cpu


In [11]:
#BLOCK 1 — Daten laden (Streaming → 200)

In [12]:
import pickle
import os

# Check if local file exists
if os.path.exists("nq_3000.pkl"):
    print("Loading from local file...")
    with open("nq_3000.pkl", "rb") as f:
        data_3000 = pickle.load(f)
    print(f"Loaded {len(data_3000)} examples from local file")
else:
    print("Loading from HuggingFace (first time)...")
    ds_stream = load_dataset("natural_questions", split="train", streaming=True)
    data_3000 = list(islice(ds_stream, 3000))
    
    # Save for next time
    print("Saving locally for future runs...")
    with open("nq_3000.pkl", "wb") as f:
        pickle.dump(data_3000, f)
    print("Saved!")

Loading from local file...
Loaded 3000 examples from local file


In [13]:
#BLOCK 2 — NQ Helper (Passagen extrahieren)

In [14]:
def extract_question(ex):
    q = ex.get("question")
    return q.get("text") if isinstance(q, dict) else q

def tokens_list(ex):
    toks = ex.get("document", {}).get("tokens", {})
    return list(toks.get("token", [])) if isinstance(toks, dict) else []

def long_answer_candidates(ex, limit=400):
    lac = ex.get("long_answer_candidates", {})
    if not isinstance(lac, dict):
        return []
    starts = list(lac.get("start_token", []))
    ends   = list(lac.get("end_token", []))
    top    = list(lac.get("top_level", []))

    n = min(len(starts), len(ends), len(top), limit)
    return [{"start_token": starts[i], "end_token": ends[i], "top_level": top[i]} for i in range(n)]

def extract_passages(ex, max_passages=2, min_words=40):
    toks = tokens_list(ex)
    passages = []

    for c in long_answer_candidates(ex, limit=800):
        if c.get("top_level") is False:
            continue

        s, e = int(c["start_token"]), int(c["end_token"])
        if e <= s:
            continue

        s = max(0, s); e = min(len(toks), e)
        if e <= s:
            continue

        text = " ".join(toks[s:e]).strip()
        if len(text.split()) >= min_words:
            passages.append(text)

        if len(passages) >= max_passages:
            break

    return passages


In [15]:
#BLOCK 3 —  DPR Retriever bauen (FAISS)

In [16]:
import os
import torch
from haystack.document_stores import FAISSDocumentStore
from haystack.nodes import DensePassageRetriever
from haystack import Document
from datetime import datetime

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
BASE_DIR = f"haystack_dpr_store_{RUN_ID}"


os.makedirs(BASE_DIR, exist_ok=True)

# 1) FRISCHEN FAISS-Store erstellen (KEIN faiss_index_path hier!)
doc_store = FAISSDocumentStore(
    sql_url=f"sqlite:///{BASE_DIR}/dpr_docs.db",
    faiss_index_factory_str="Flat",
    similarity="dot_product",
    return_embedding=True,
    validate_index_sync=False
)

# 2) Dokumente bauen
docs = []
for i, ex in enumerate(data_3000):
    for j, psg in enumerate(extract_passages(ex, max_passages=2)):
        docs.append(Document(content=psg, meta={"ex_id": i, "p_id": j}))

doc_store.write_documents(docs)

# 3) DPR Retriever
DPR_QUERY = "facebook/dpr-question_encoder-single-nq-base"
DPR_CTX   = "facebook/dpr-ctx_encoder-single-nq-base"

retriever = DensePassageRetriever(
    document_store=doc_store,
    query_embedding_model=DPR_QUERY,
    passage_embedding_model=DPR_CTX,
    max_seq_len_query=256,
    max_seq_len_passage=256,
    batch_size=16,
    use_gpu=torch.cuda.is_available(),
    embed_title=False
)

# 4) Embeddings erzeugen
doc_store.update_embeddings(retriever, batch_size=16)

# 5) OPTIONAL: Speichern (damit du später NICHT neu embeddest)
doc_store.save(
    index_path=f"{BASE_DIR}/dpr_index.faiss",
    config_path=f"{BASE_DIR}/dpr_config.json"
)

print("Docs:", doc_store.get_document_count())
print("Embeddings:", doc_store.get_embedding_count())


Writing Documents: 10000it [00:14, 673.88it/s]                                                                         
Create embeddings: 100%|████████████████████████████████████████████████████████████| 16/16 [00:03<00:00,  4.38 Docs/s]
Documents Processed: 5648 docs [28:50,  3.26 docs/s]                                                                   

Docs: 5646
Embeddings: 5646


In [17]:
#BLOCK 4 — Generator laden (FLAN-T5 auf GPU)

In [18]:
MODEL_NAME = "google/flan-t5-base"  # falls VRAM knapp: "google/flan-t5-small"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
gen = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

print("Generator loaded:", MODEL_NAME)


Device: cpu
Generator loaded: google/flan-t5-base


In [19]:
#BLOCK 5 — RAG Answer Funktion (Retriever + Generator)

In [20]:
def build_context(retrieved_docs, max_chars=3000):
    ctx = ""
    for d in retrieved_docs:
        chunk = d.content
        if len(ctx) + len(chunk) + 2 > max_chars:
            break
        ctx += chunk + "\n\n"
    return ctx.strip()

def rag_answer(question, top_k=10, max_chars=3000, max_new_tokens=64):
    # 1) RETRIEVAL
    retrieved = retriever.retrieve(question, top_k=top_k)

    # 2) CONTEXT BUILDING
    context = build_context(retrieved, max_chars=max_chars)

    # 3) PROMPT
    prompt = f"""Answer the question using ONLY the context.
If the context does not contain the answer, say: "I don't know."

Question: {question}

Context:
{context}

Answer:"""

    # 4) GENERATION
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    out = gen.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    answer = tok.decode(out[0], skip_special_tokens=True)

    return answer, retrieved, context


In [23]:
#BLOCK 7 — Evaluation Start (Gold + EM)

In [24]:
# -------------------------
# BLOCK 7 — GOLD SHORT ANSWER EXTRACTION (FIXED for list/dict annotations)
# -------------------------

def extract_nq_short_answer(example, max_words=10):
    """
    Extract a short answer string from a Natural Questions example.
    Supports different HF schemas (answers / short_answers / annotations as list or dict).
    Returns a short string or None.
    """

    # Case 1: HF-style answers field
    if "answers" in example and example["answers"]:
        ans = example["answers"]
        if isinstance(ans, list) and ans:
            if isinstance(ans[0], str):
                txt = ans[0].strip()
                if txt and len(txt.split()) <= max_words:
                    return txt
        if isinstance(ans, dict) and "text" in ans:
            txts = ans["text"]
            if isinstance(txts, list) and txts:
                txt = txts[0].strip()
                if txt and len(txt.split()) <= max_words:
                    return txt

    # Case 2: short_answers field
    if "short_answers" in example and example["short_answers"]:
        sa = example["short_answers"]
        if isinstance(sa, list) and sa:
            first = sa[0]
        else:
            first = sa

        if isinstance(first, str):
            txt = first.strip()
            if txt and len(txt.split()) <= max_words:
                return txt

        if isinstance(first, dict) and "text" in first:
            t = first["text"]
            if isinstance(t, str):
                txt = t.strip()
                if txt and len(txt.split()) <= max_words:
                    return txt
            if isinstance(t, list) and t:
                txt = t[0].strip()
                if txt and len(txt.split()) <= max_words:
                    return txt

    # Case 3: annotations (can be list OR dict)
    if "annotations" in example and example["annotations"]:
        ann = example["annotations"]
        if isinstance(ann, list) and ann:
            ann0 = ann[0]
        else:
            ann0 = ann  # dict case

        if isinstance(ann0, dict) and "short_answers" in ann0 and ann0["short_answers"]:
            sas = ann0["short_answers"]
            if isinstance(sas, list) and sas:
                sa0 = sas[0]
            else:
                sa0 = sas

            if isinstance(sa0, dict) and "text" in sa0 and sa0["text"]:
                t = sa0["text"]
                if isinstance(t, str):
                    txt = t.strip()
                    if txt and len(txt.split()) <= max_words:
                        return txt
                if isinstance(t, list) and t:
                    txt = t[0].strip()
                    if txt and len(txt.split()) <= max_words:
                        return txt

    return None


# Build gold answers and covered indices
golds = [extract_nq_short_answer(ex) for ex in data_3000]
covered_idx = [i for i, g in enumerate(golds) if isinstance(g, str) and g.strip() != ""]

print("Coverage:", len(covered_idx) / len(data_3000))
print("Example gold answers:", [golds[i] for i in covered_idx[:10]])


Coverage: 0.32366666666666666
Example gold answers: ['March 18, 2018', 'Persephone (/pərˈsɛfəni/; Greek: Περσεφόνη), also called Kore (/ˈkɔːriː/; "the maiden")', 'the Shulchan Aruch', 'Real Madrid', '1983', 'Icona Pop and Charli XCX', 'Shmi', '1 square hectometre (hm2)', '13th', 'Luis Guillermo Solís Rivera']


In [25]:
# -------------------------
# BLOCK 9 — TOKEN F1 (QA) [FIXED]
# -------------------------
# F1 vergleicht Token-Overlap zwischen Prediction und Gold.
# Weniger streng als EM, Standard in QA (SQuAD-style).

import re
import string

def normalize(text):
    """
    Normalize text for EM / F1:
    - lowercase
    - remove punctuation
    - normalize whitespace
    """
    if text is None:
        return ""
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def f1_score(pred, gold):
    if gold is None:
        return None

    pred_toks = normalize(pred).split()
    gold_toks = normalize(gold).split()

    if len(pred_toks) == 0 and len(gold_toks) == 0:
        return 1.0
    if len(pred_toks) == 0 or len(gold_toks) == 0:
        return 0.0

    # Token overlap (multiset / bag-of-words)
    common = {}
    for t in pred_toks:
        common[t] = common.get(t, 0) + 1

    num_same = 0
    for t in gold_toks:
        if common.get(t, 0) > 0:
            num_same += 1
            common[t] -= 1

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_toks)
    recall = num_same / len(gold_toks)
    return 2 * precision * recall / (precision + recall)


# Quick sanity test
print(
    "F1 (test example):",
    round(f1_score("February 25 , 2018", "March 18, 2018"), 4)
)


F1 (test example): 0.3333


In [26]:
# -------------------------
# BLOCK 10 — E2E QA METRICS ON GOLD SUBSET (EM + F1) [FIXED]
# -------------------------
# Wir berechnen EM und F1 nur für covered_idx (wo Gold existiert).

def exact_match(pred, gold):
    """
    Exact Match nach Normalisierung:
    - lowercase
    - punctuation entfernt
    - whitespace normalisiert
    """
    if gold is None:
        return None
    return 1.0 if normalize(pred) == normalize(gold) else 0.0


em_scores = []
f1_scores = []

k = 10  # konsistent zu deinem Setup (wenn du überall k=10 willst)

for i in covered_idx:
    q = extract_question(data_3000[i])
    pred, _, _ = rag_answer(q, top_k=k)

    gold = golds[i]
    em = exact_match(pred, gold)
    f1 = f1_score(pred, gold)

    if em is not None:
        em_scores.append(em)
    if f1 is not None:
        f1_scores.append(f1)

print("Evaluated (gold subset):", len(covered_idx))
print("EM mean:", round(sum(em_scores) / len(em_scores), 4) if em_scores else None)
print("F1 mean:", round(sum(f1_scores) / len(f1_scores), 4) if f1_scores else None)


Evaluated (gold subset): 971
EM mean: 0.2276
F1 mean: 0.313


In [27]:
# -------------------------
# BLOCK 11 — RETRIEVER METRICS (Recall@k, Precision@k, MRR@k)  [EXTENDED]
# -------------------------

def gold_in_doc(gold, doc_text):
    """1 wenn Gold (normalisiert) als Substring im doc_text vorkommt, sonst 0."""
    if gold is None:
        return None
    g = normalize(gold)
    if not g:
        return None
    t = normalize(doc_text)
    return int(g in t)

def eval_retrieval_metrics_on_gold_subset(k_list=(1,3,5,10)):
    """
    Proxy-Retrieval-Metriken auf dem Gold-Subset (covered_idx):
    - Recall@k: mindestens 1 relevanter Doc in Top-k
    - Precision@k: Anteil relevanter Docs in Top-k
    - MRR@k: 1/rank des ersten relevanten Docs in Top-k, sonst 0
    """
    max_k = max(k_list)

    # Sammeln pro k
    recall_hits = {k: [] for k in k_list}
    precision_vals = {k: [] for k in k_list}
    mrr_vals = {k: [] for k in k_list}

    for i in covered_idx:
        q = extract_question(data_3000[i])
        gold = golds[i]

        retrieved = retriever.retrieve(q, top_k=max_k)

        # doc-wise relevance (0/1) für max_k
        rel = []
        for d in retrieved:
            h = gold_in_doc(gold, d.content)
            if h is None:
                rel = None
                break
            rel.append(h)

        if rel is None:
            continue

        for k in k_list:
            rel_k = rel[:k]

            # Recall@k
            recall_hits[k].append(int(any(rel_k)))

            # Precision@k
            precision_vals[k].append(sum(rel_k) / k)

            # MRR@k
            rr = 0.0
            for rank, hit in enumerate(rel_k, start=1):
                if hit == 1:
                    rr = 1.0 / rank
                    break
            mrr_vals[k].append(rr)

    # Ergebnisse
    out = {}
    n = len(recall_hits[k_list[0]])
    out["n_eval"] = n

    for k in k_list:
        out[f"recall@{k}"] = (sum(recall_hits[k]) / len(recall_hits[k])) if recall_hits[k] else None
        out[f"precision@{k}"] = (sum(precision_vals[k]) / len(precision_vals[k])) if precision_vals[k] else None
        out[f"mrr@{k}"] = (sum(mrr_vals[k]) / len(mrr_vals[k])) if mrr_vals[k] else None

    return out

metrics = eval_retrieval_metrics_on_gold_subset(k_list=(1,3,5,10))

print("Retriever metrics (proxy) on gold subset")
print("n_eval:", metrics["n_eval"])
for k in (1,3,5,10):
    print(
        f"@{k}:",
        "Recall", round(metrics[f"recall@{k}"], 4) if metrics[f"recall@{k}"] is not None else None,
        "| Precision", round(metrics[f"precision@{k}"], 4) if metrics[f"precision@{k}"] is not None else None,
        "| MRR", round(metrics[f"mrr@{k}"], 4) if metrics[f"mrr@{k}"] is not None else None,
    )


Retriever metrics (proxy) on gold subset
n_eval: 971
@1: Recall 0.3986 | Precision 0.3986 | MRR 0.3986
@3: Recall 0.4964 | Precision 0.2025 | MRR 0.4427
@5: Recall 0.5191 | Precision 0.1392 | MRR 0.4479
@10: Recall 0.5561 | Precision 0.0867 | MRR 0.4527


In [28]:
# -------------------------
# BLOCK 12 — END-TO-END EVAL + PER-QUERY OUTPUTS (k=10)
# -------------------------
import pandas as pd
import evaluate

rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

k_list = (1, 3, 5, 10)
max_k = max(k_list)

em_scores = []
f1_scores = []
rougeL_scores = []
bleu_scores = []

per_query_rows = []


for i in covered_idx:
    q = extract_question(data_3000[i])
    gold = golds[i]

    pred, retrieved, _ = rag_answer(q, top_k=max_k)

    # ---------- Proxy relevance per retrieved doc ----------
    g = normalize(gold) if gold is not None else ""
    rel = []
    if g:
        for d in retrieved[:max_k]:
            rel.append(1 if g in normalize(d.content) else 0)
    else:
        rel = [0] * min(len(retrieved), max_k)

    # ---------- hit / precision / rr per k ----------
    hit_at = {}
    prec_at = {}
    rr_at = {}

    for kk in k_list:
        rel_k = rel[:kk]
        hit_at[kk] = int(any(rel_k)) if rel_k else 0
        prec_at[kk] = (sum(rel_k) / kk) if rel_k else 0.0

        rr = 0.0
        for rank, h in enumerate(rel_k, start=1):
            if h == 1:
                rr = 1.0 / rank
                break
        rr_at[kk] = rr

    # ---------- QA metrics ----------
    em = exact_match(pred, gold)
    f1 = f1_score(pred, gold)

    # ---------- ROUGE / BLEU ----------
    rougeL = rouge_metric.compute(predictions=[pred], references=[gold])["rougeL"]

    # Safe BLEU calculation (handles empty predictions)
    try:
        if pred and len(pred.split()) > 0:  # Check if prediction is not empty
            bleu = bleu_metric.compute(predictions=[pred], references=[[gold]])["bleu"]
        else:
            bleu = 0.0
    except (ZeroDivisionError, ValueError):
        bleu = 0.0
    # ---------- collect means ----------
    if em is not None:
        em_scores.append(em)
    if f1 is not None:
        f1_scores.append(f1)

    rougeL_scores.append(rougeL)
    bleu_scores.append(bleu)

    # ---------- per-query row ----------
    per_query_rows.append({
        "idx": i,
        "question": q,
        "gold": gold,
        "pred": pred,

        "hit@1": hit_at[1],
        "hit@3": hit_at[3],
        "hit@5": hit_at[5],
        "hit@10": hit_at[10],

        "precision@1": prec_at[1],
        "precision@3": prec_at[3],
        "precision@5": prec_at[5],
        "precision@10": prec_at[10],

        "rr@1": rr_at[1],
        "rr@3": rr_at[3],
        "rr@5": rr_at[5],
        "rr@10": rr_at[10],

        "em": em,
        "f1": f1,
        "rougeL": rougeL,
        "bleu": bleu,
    })

# ---------- Debug overview ----------
df_dbg = pd.DataFrame(per_query_rows)

print("per_query_rows:", len(per_query_rows))
print("EM mean:", sum(em_scores) / len(em_scores) if em_scores else None)
print("F1 mean:", sum(f1_scores) / len(f1_scores) if f1_scores else None)
print("ROUGE-L mean:", sum(rougeL_scores) / len(rougeL_scores) if rougeL_scores else None)
print("BLEU mean:", sum(bleu_scores) / len(bleu_scores) if bleu_scores else None)

df_dbg.head(10)


per_query_rows: 971
EM mean: 0.22760041194644695
F1 mean: 0.313021862421001
ROUGE-L mean: 0.3145622978440317
BLEU mean: 0.04092366110817759


,idx,question,gold,pred,hit@1,hit@3,hit@5,hit@10,precision@1,precision@3,precision@5,precision@10,rr@1,rr@3,rr@5,rr@10,em,f1,rougeL,bleu
0,0,when is the last episode of season 8 of the wa...,"March 18, 2018","February 25, 2018",0,0,0,0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.333333,0.333333,0.0
1,1,in greek mythology who was the goddess of spri...,"Persephone (/pərˈsɛfəni/; Greek: Περσεφόνη), a...",cherry blossom,0,0,0,0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0
2,6,what is the name of the most important jewish ...,the Shulchan Aruch,The Torah,0,0,0,0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.400000,0.400000,0.0
3,8,what is the name of spain's most famous soccer...,Real Madrid,Real Madrid,0,0,1,1,0.0,0.000000,0.2,0.3,0.0,0.0,0.2,0.2,1.0,1.000000,1.000000,0.0
4,9,when was the first robot used in surgery,1983,I don't know,0,0,0,0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0
5,13,who sings the song i don't care i love it,Icona Pop and Charli XCX,Icona Pop,0,0,0,0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.571429,0.571429,0.0
6,16,who are uncle owen and aunt beru related to,Shmi,Luke Skywalker,0,0,0,0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0
7,20,100 acres is equal to how many hectares,1 square hectometre (hm2),0.405,0,0,0,0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0
8,22,where was donovan mitchell picked in the draft,13th,Denver Nuggets,1,1,1,1,1.0,0.666667,0.4,0.2,1.0,1.0,1.0,1.0,0.0,0.000000,0.000000,0.0
9,27,who is the president of costa rica 2017,Luis Guillermo Solís Rivera,Ram Nath Kovind,1,1,1,1,1.0,0.333333,0.2,0.1,1.0,1.0,1.0,1.0,0.0,0.000000,0.000000,0.0


In [29]:
# -------------------------
# BLOCK 13 — SAVE METRICS (CSV) [FINAL]
# -------------------------
# Speichert:
# - Coverage
# - EM mean
# - F1 mean
# - ROUGE-L mean
# - BLEU mean
# - Recall@k, Precision@k, MRR@k (Proxy, aus Block 11)

import pandas as pd

# Robust means (falls Listen None enthalten)
em_valid = [x for x in em_scores if x is not None]
f1_valid = [x for x in f1_scores if x is not None]
rouge_valid = [x for x in rougeL_scores if x is not None]
bleu_valid = [x for x in bleu_scores if x is not None]

em_mean = sum(em_valid) / len(em_valid) if em_valid else None
f1_mean = sum(f1_valid) / len(f1_valid) if f1_valid else None
rougeL_mean = sum(rouge_valid) / len(rouge_valid) if rouge_valid else None
bleu_mean = sum(bleu_valid) / len(bleu_valid) if bleu_valid else None

coverage = len(covered_idx) / len(data_3000) if len(data_3000) else None

results = {
    "metric": [
        "coverage",
        "em_mean",
        "f1_mean",
        "rougeL_mean",
        "bleu_mean",
        "n_eval_retrieval",
        "recall@1", "precision@1", "mrr@1",
        "recall@3", "precision@3", "mrr@3",
        "recall@5", "precision@5", "mrr@5",
        "recall@10","precision@10","mrr@10",
    ],
    "value": [
        coverage,
        em_mean,
        f1_mean,
        rougeL_mean,
        bleu_mean,
        metrics.get("n_eval"),
        metrics.get("recall@1"),  metrics.get("precision@1"),  metrics.get("mrr@1"),
        metrics.get("recall@3"),  metrics.get("precision@3"),  metrics.get("mrr@3"),
        metrics.get("recall@5"),  metrics.get("precision@5"),  metrics.get("mrr@5"),
        metrics.get("recall@10"), metrics.get("precision@10"), metrics.get("mrr@10"),
    ],
}

df_results = pd.DataFrame(results)
df_results


,metric,value
0,coverage,0.323667
1,em_mean,0.227600
2,f1_mean,0.313022
3,rougeL_mean,0.314562
4,bleu_mean,0.040924
5,n_eval_retrieval,971.000000
6,recall@1,0.398558
7,precision@1,0.398558
8,mrr@1,0.398558
9,recall@3,0.496395


In [30]:
# -------------------------
# BLOCK 14 — SAVE PER-QUERY CSV
# -------------------------
import pandas as pd

df_per_query = pd.DataFrame(per_query_rows)

# Labels setzen (pro Notebook passend)
df_per_query["retriever"] = "DPR"    
df_per_query["generator"] = "T5"      

out_path = "rag_outputs_DPR_T5_per_query_3000.csv"  # pro Notebook anpassen
df_per_query.to_csv(out_path, index=False)

print("Saved per-query results:", out_path)
df_per_query.head()
df_results.to_csv("rag_metrics_dpr_flan_t5_3000.csv", index=False)


Saved per-query results: rag_outputs_DPR_T5_per_query_3000.csv
